# Optimization Modelling — Task 2

Realistic capacity expansion with piecewise marginal costs and minimum distance constraints between facilities.


In [47]:
# Clear all old CSVs first
import os
for f in os.listdir('/content'):
    if f.endswith('.csv'):
        os.remove(f'/content/{f}')
print("Cleared old CSVs")

Cleared old CSVs


## 1. Imports

In [48]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from google.colab import files
uploaded = files.upload()

Saving avg_income_nyc.csv to avg_income_nyc.csv
Saving child_care_regulated_nyc.csv to child_care_regulated_nyc.csv
Saving employment_rate_nyc.csv to employment_rate_nyc.csv
Saving population_nyc.csv to population_nyc.csv
Saving potential_locations_nyc.csv to potential_locations_nyc.csv


In [49]:
import os
print(os.listdir('/content'))

['.config', 'gurobi.lic', 'potential_locations_nyc.csv', 'population_nyc.csv', 'child_care_regulated_nyc.csv', 'avg_income_nyc.csv', 'employment_rate_nyc.csv', 'sample_data']


In [50]:
import os
# Remove any old license files
for f in os.listdir('/content'):
    if 'gurobi' in f.lower() and '.lic' in f.lower():
        os.remove(f'/content/{f}')
        print(f"Removed: {f}")
print("Done")

Removed: gurobi.lic
Done


In [51]:
from google.colab import files
uploaded = files.upload()  # select the gurobi.lic you just downloaded

import os
os.environ['GRB_LICENSE_FILE'] = '/content/gurobi.lic'

Saving gurobi.lic to gurobi.lic


In [52]:
import gurobipy as gp
m_test = gp.Model()
m_test.addVars(10000, vtype='B')
m_test.update()
print("License OK — variables:", m_test.NumVars)
m_test.dispose()

License OK — variables: 10000


## 2. Load Data

- **Task 1** — population and income/employment (already cleaned and ZIP-aligned)
- **Task 2** — ZIP-aligned child care facilities and potential new locations (with coordinates)


In [53]:
population_nyc           = pd.read_csv('/content/population_nyc.csv')
avg_income_nyc           = pd.read_csv('/content/avg_income_nyc.csv')
employment_rate_nyc      = pd.read_csv('/content/employment_rate_nyc.csv')
child_care_regulated_nyc = pd.read_csv('/content/child_care_regulated_nyc.csv')
potential_locations_nyc  = pd.read_csv('/content/potential_locations_nyc.csv')

In [54]:
pop = population_nyc.rename(columns={'-5': 'pop_0_5', '6-12': 'pop_6_12'})[['zipcode', 'pop_0_5', 'pop_6_12']]
pop['pop_2wk_12yr'] = pop['pop_0_5'] + pop['pop_6_12']
pop.head()

,zipcode,pop_0_5,pop_6_12,pop_2wk_12yr
0,10001,744,1255,1999
1,10002,2142,4645,6787
2,10003,1440,1510,2950
3,10004,433,262,695
4,10005,484,318,802


In [55]:
child_care_regulated_nyc['slots_0_5'] = (
    child_care_regulated_nyc['infant_capacity'] +
    child_care_regulated_nyc['toddler_capacity'] +
    child_care_regulated_nyc['preschool_capacity']
)

slots = child_care_regulated_nyc.groupby('zipcode').agg(
    total_slots=('total_capacity', 'sum'),
    slots_0_5=('slots_0_5', 'sum')
).reset_index()
slots.head()

,zipcode,total_slots,slots_0_5
0,10001,609,0
1,10002,4729,18
2,10003,1995,0
3,10004,263,0
4,10005,39,0


In [56]:
df = pop.merge(slots, on='zipcode', how='left')
df = df.merge(avg_income_nyc.rename(columns={'average income': 'avg_income'}), on='zipcode', how='left')
df = df.merge(employment_rate_nyc.rename(columns={'employment rate': 'emp_rate'}), on='zipcode', how='left')
df[['total_slots', 'slots_0_5']] = df[['total_slots', 'slots_0_5']].fillna(0)
df.head()

,zipcode,pop_0_5,pop_6_12,pop_2wk_12yr,total_slots,slots_0_5,avg_income,emp_rate
0,10001,744,1255,1999,609.0,0.0,102878.033603,0.595097
1,10002,2142,4645,6787,4729.0,18.0,59604.041165,0.520662
2,10003,1440,1510,2950,1995.0,0.0,114273.049645,0.497244
3,10004,433,262,695,263.0,0.0,132004.310345,0.506661
4,10005,484,318,802,39.0,0.0,121437.713311,0.665833


In [57]:
df['demand'] = df.apply(
    lambda r: 'High' if (r['avg_income'] <= 60_000 or r['emp_rate'] >= 0.60) else 'Normal',
    axis=1
)
print(df['demand'].value_counts().to_string())
df[['zipcode', 'avg_income', 'emp_rate', 'demand']].head(10)

demand
High      97
Normal    83


,zipcode,avg_income,emp_rate,demand
0,10001,102878.033603,0.595097,Normal
1,10002,59604.041165,0.520662,High
2,10003,114273.049645,0.497244,Normal
3,10004,132004.310345,0.506661,Normal
4,10005,121437.713311,0.665833,High
5,10006,126377.118644,0.631692,High
6,10007,138853.904282,0.528910,Normal
7,10009,77133.233533,0.514567,Normal
8,10010,116272.698810,0.492749,Normal
9,10011,120420.792079,0.557000,Normal


In [58]:
THRESHOLD_HIGH   = 1/2
THRESHOLD_NORMAL = 1/3
THRESHOLD_UNDER5 = 2/3

df['total_threshold'] = df.apply(
    lambda r: THRESHOLD_HIGH * r['pop_2wk_12yr'] if r['demand'] == 'High'
              else THRESHOLD_NORMAL * r['pop_2wk_12yr'],
    axis=1
)

df['fail_total']  = df['total_slots'] <= df['total_threshold']
df['fail_under5'] = df['slots_0_5']   <= THRESHOLD_UNDER5 * df['pop_0_5']
df['is_desert']   = (df['fail_total'] | df['fail_under5']) & (df['pop_2wk_12yr'] > 0)
print(f"Zips failing total capacity check : {df['fail_total'].sum()}")
print(f"Zips failing under-5 policy check : {df['fail_under5'].sum()}")
print(f"Total desert zip codes            : {df['is_desert'].sum()}")

Zips failing total capacity check : 162
Zips failing under-5 policy check : 180
Total desert zip codes            : 179


In [59]:
df['total_slot_deficit'] = np.ceil(
    (df['total_threshold'] - df['total_slots']).clip(lower=0)
).astype(int)

df['under5_slot_deficit'] = np.ceil(
    (THRESHOLD_UNDER5 * df['pop_0_5'] - df['slots_0_5']).clip(lower=0)
).astype(int)
df[['zipcode', 'total_slots', 'total_threshold', 'total_slot_deficit',
    'slots_0_5', 'under5_slot_deficit']].head(10)

,zipcode,total_slots,total_threshold,total_slot_deficit,slots_0_5,under5_slot_deficit
0,10001,609.0,666.333333,58,0.0,496
1,10002,4729.0,3393.500000,0,18.0,1410
2,10003,1995.0,983.333333,0,0.0,960
3,10004,263.0,231.666667,0,0.0,289
4,10005,39.0,401.000000,362,0.0,323
5,10006,156.0,130.500000,0,14.0,72
6,10007,284.0,400.333333,117,0.0,404
7,10009,1784.0,1490.333333,0,18.0,1246
8,10010,234.0,1162.666667,929,0.0,948
9,10011,1956.0,1030.333333,0,42.0,764


In [60]:
# ── Distance Preprocessing ─────────────────────────────────────────────────────
# Task 2 requires that no two facilities (existing or new) within the same ZIP
# are closer than 0.06 miles. We pre-compute this BEFORE building the Gurobi model
# so the model doesn't need to compute distances itself.
#
# This produces two outputs:
#   infeasible_locs : potential locations that are already too close to an
#                     existing facility → cannot be built at all
#   forbidden_pairs : pairs of potential locations too close to each other
#                     → at most one of them can be selected

MIN_DIST = 0.06    # minimum distance in miles
R_EARTH  = 3958.8  # Earth radius in miles

def haversine(lat1, lon1, lat2, lon2):
    """
    Vectorized Haversine distance matrix (miles).
    Returns D where D[i, j] = distance(point2[i], point1[j]).
    """
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    dlat = lat2[:, None] - lat1[None, :]
    dlon = lon2[:, None] - lon1[None, :]
    a    = (np.sin(dlat/2)**2
            + np.cos(lat1[None, :]) * np.cos(lat2[:, None]) * np.sin(dlon/2)**2)
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

In [61]:
# ── Prepare datasets ───────────────────────────────────────────────────────────
desert_zips = df[df['is_desert']]['zipcode'].tolist()

# Existing facilities in desert ZIPs (drop zero-capacity — same as Task 1)
desert_facilities = child_care_regulated_nyc[
    child_care_regulated_nyc['zipcode'].isin(desert_zips) &
    (child_care_regulated_nyc['total_capacity'] > 0)
].copy().reset_index(drop=True)

# Potential locations — assign a unique integer loc_id = row index
pot_locs = potential_locations_nyc.copy().reset_index(drop=True)
pot_locs['loc_id'] = pot_locs.index

print(f"Desert ZIPs                : {len(desert_zips)}")
print(f"Existing facilities        : {len(desert_facilities)}")
print(f"Total potential locations  : {len(pot_locs)}")

Desert ZIPs                : 179
Existing facilities        : 7739
Total potential locations  : 31100


In [62]:
# ── Compute infeasible locations and forbidden pairs ───────────────────────────
infeasible_locs = set()  # loc_ids too close to an existing facility → cannot build here
forbidden_pairs = set()  # (loc_id_i, loc_id_j) → cannot build at BOTH locations

for z in desert_zips:
    ex = desert_facilities[desert_facilities['zipcode'] == z]
    pl = pot_locs[pot_locs['zipcode'] == z]

    if len(pl) == 0:
        continue

    pl_ids  = pl['loc_id'].values
    pl_lats = pl['latitude'].values
    pl_lons = pl['longitude'].values

    # ── 1. Existing facility vs Potential location ─────────────────────────────
    # If a potential location is within MIN_DIST of ANY existing facility
    # in the same ZIP → it is infeasible (we cannot build there)
    if len(ex) > 0:
        dist_ep = haversine(
            ex['latitude'].values, ex['longitude'].values,
            pl_lats, pl_lons
        )  # shape: (len(pl), len(ex))
        too_close = np.any(dist_ep < MIN_DIST, axis=1)  # True for each bad potential loc
        for loc_id, bad in zip(pl_ids, too_close):
            if bad:
                infeasible_locs.add(int(loc_id))

    # ── 2. Potential location vs Potential location ────────────────────────────
    # If two potential locations are within MIN_DIST of each other
    # → at most one can be chosen (forbidden pair constraint in the model)
    if len(pl) >= 2:
        dist_pp = haversine(pl_lats, pl_lons, pl_lats, pl_lons)
        # Upper triangle only (i < j) to avoid duplicates
        rows, cols = np.where(
            (dist_pp < MIN_DIST) &
            np.triu(np.ones_like(dist_pp, dtype=bool), k=1)
        )
        for i, j in zip(rows, cols):
            forbidden_pairs.add((int(pl_ids[i]), int(pl_ids[j])))

# Remove forbidden pairs that involve infeasible locations (already ruled out)
forbidden_pairs = {
    (i, j) for i, j in forbidden_pairs
    if i not in infeasible_locs and j not in infeasible_locs
}

# Final feasible potential locations
feasible_locs = pot_locs[~pot_locs['loc_id'].isin(infeasible_locs)].copy()

print(f"Total potential locations                          : {len(pot_locs)}")
print(f"Infeasible (too close to existing facility)       : {len(infeasible_locs)}")
print(f"Feasible potential locations remaining             : {len(feasible_locs)}")
print(f"Forbidden pairs among feasible potential locations : {len(forbidden_pairs)}")

Total potential locations                          : 31100
Infeasible (too close to existing facility)       : 3141
Feasible potential locations remaining             : 27959
Forbidden pairs among feasible potential locations : 86996


In [63]:
import gurobipy as gp
from gurobipy import GRB

# ── Cost / capacity parameters (same as Task 1) ───────────────────────────────
COST_S = 65_000;  CAP_S = 100;  UNDER5_CAP_S = 50
COST_M = 95_000;  CAP_M = 200;  UNDER5_CAP_M = 100
COST_L = 115_000; CAP_L = 400;  UNDER5_CAP_L = 200
EQUIP_COST = 100

# ── Index sets ─────────────────────────────────────────────────────────────────
cap_lookup  = desert_facilities.set_index('facility_id')['total_capacity'].to_dict()
fac_by_zip  = desert_facilities.groupby('zipcode')['facility_id'].apply(list).to_dict()
desert_df   = df[df['is_desert']].set_index('zipcode')

# Feasible locations per ZIP
feasible_locs['slots_0_5'] = (
    feasible_locs.get('infant_capacity', 0) +
    feasible_locs.get('toddler_capacity', 0) +
    feasible_locs.get('preschool_capacity', 0)
)
locs_by_zip = feasible_locs.groupby('zipcode')['loc_id'].apply(list).to_dict()

# ── Gurobi model ───────────────────────────────────────────────────────────────
m = gp.Model("ChildCareDesert_Task2")
m.setParam('OutputFlag', 1)

# ── Variable 1a: x_05[f] — 0-5 expansion slots at existing facility f ─────────
x_05  = m.addVars(list(cap_lookup.keys()), vtype=GRB.INTEGER, lb=0, name="x_05")

# ── Variable 1b: x_512[f] — 5-12 expansion slots at existing facility f ───────
x_512 = m.addVars(list(cap_lookup.keys()), vtype=GRB.INTEGER, lb=0, name="x_512")

# ── Variable 2: b[loc_id, size] — binary: build a facility of given size
#    at feasible location loc_id ─────────────────────────────────────────────────
all_loc_ids = feasible_locs['loc_id'].tolist()
b = {}
for size in ['S', 'M', 'L']:
    b[size] = m.addVars(all_loc_ids, vtype=GRB.BINARY, name=f"b_{size}")

# ── Variable 3a: n_05[z] — 0-5 slots from new builds in desert ZIP z ──────────
n_05  = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name="n_05")

# ── Variable 3b: n_512[z] — 5-12 slots from new builds in desert ZIP z ────────
n_512 = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name="n_512")

m.update()
print("Decision variables defined:")
print(f"  x_05     : {len(x_05):>6}  (0-5  expansion slots per existing facility)")
print(f"  x_512    : {len(x_512):>6}  (5-12 expansion slots per existing facility)")
print(f"  b_S      : {len(b['S']):>6}  (binary: build small  facility at location)")
print(f"  b_M      : {len(b['M']):>6}  (binary: build medium facility at location)")
print(f"  b_L      : {len(b['L']):>6}  (binary: build large  facility at location)")
print(f"  n_05     : {len(n_05):>6}  (0-5  slots from new builds per desert ZIP)")
print(f"  n_512    : {len(n_512):>6}  (5-12 slots from new builds per desert ZIP)")
print(f"\nTotal variables: {m.NumVars}")

Set parameter OutputFlag to value 1
Decision variables defined:
  x_05     :   7739  (0-5  expansion slots per existing facility)
  x_512    :   7739  (5-12 expansion slots per existing facility)
  b_S      :  27959  (binary: build small  facility at location)
  b_M      :  27959  (binary: build medium facility at location)
  b_L      :  27959  (binary: build large  facility at location)
  n_05     :    179  (0-5  slots from new builds per desert ZIP)
  n_512    :    179  (5-12 slots from new builds per desert ZIP)

Total variables: 99713


In [64]:
# ── Piecewise expansion cost auxiliaries ──────────────────────────────────────
# 3 binary variables per facility select which tier the expansion falls in:
#   t1[f]: expansion in (0%,  10%] of n_f  → cheapest tier
#   t2[f]: expansion in (10%, 15%] of n_f  → middle tier
#   t3[f]: expansion in (15%, 20%] of n_f  → expensive tier
# Each tier uses a separate continuous variable for the slots in that tier:
#   s1[f], s2[f], s3[f]

t1 = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,     name="t1")
t2 = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,     name="t2")
t3 = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,     name="t3")
s1 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="s1")
s2 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="s2")
s3 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="s3")

for f, n_f in cap_lookup.items():
    b1 = 0.10 * n_f   # 10% breakpoint
    b2 = 0.15 * n_f   # 15% breakpoint
    b3 = 0.20 * n_f   # 20% breakpoint (hard cap)

    x_f = x_05[f] + x_512[f]  # total expansion at facility f

    # Exactly one tier must be active (or none if x_f = 0)
    # We use: t1 + t2 + t3 <= 1, and link x_f = s1 + s2 + s3
    m.addConstr(t1[f] + t2[f] + t3[f] <= 1,       name=f"one_tier_{f}")
    m.addConstr(x_f == s1[f] + s2[f] + s3[f],     name=f"slot_split_{f}")

    # Tier bounds: s_k is nonzero only when t_k = 1
    m.addConstr(s1[f] <= b1 * t1[f],              name=f"s1_ub_{f}")
    m.addConstr(s2[f] <= (b2 - b1) * t2[f],       name=f"s2_ub_{f}")
    m.addConstr(s3[f] <= (b3 - b2) * t3[f],       name=f"s3_ub_{f}")

    # Tier activation: force higher tiers only when lower tiers are full
    m.addConstr(s1[f] >= b1 * t2[f],              name=f"s1_full_if_t2_{f}")
    m.addConstr(s1[f] >= b1 * t3[f],              name=f"s1_full_if_t3_{f}")
    m.addConstr(s2[f] >= (b2 - b1) * t3[f],       name=f"s2_full_if_t3_{f}")

    # Hard cap: total expansion <= 20% of current capacity
    m.addConstr(x_f <= b3,                         name=f"exp_cap_{f}")

# ── Expansion cost per facility ────────────────────────────────────────────────
# Tier 1: (20,000 + 200·n_f) · s1/n_f
# Tier 2: (20,000 + 400·n_f) · s2/n_f
# Tier 3: (20,000 + 1000·n_f) · s3/n_f
C_exp = gp.quicksum(
    (20_000 + 200  * cap_lookup[f]) / cap_lookup[f] * s1[f]
  + (20_000 + 400  * cap_lookup[f]) / cap_lookup[f] * s2[f]
  + (20_000 + 1000 * cap_lookup[f]) / cap_lookup[f] * s3[f]
    for f in cap_lookup
)

# ── Construction cost ──────────────────────────────────────────────────────────
C_build = gp.quicksum(
    COST_S * b['S'][l] + COST_M * b['M'][l] + COST_L * b['L'][l]
    for l in all_loc_ids
)

# ── Equipment surcharge ($100 per 0-5 slot added) ─────────────────────────────
C_equip = EQUIP_COST * gp.quicksum(
    gp.quicksum(x_05[f] for f in fac_by_zip.get(z, []))
    + n_05[z]
    for z in desert_zips
)

m.setObjective(C_exp + C_build + C_equip, GRB.MINIMIZE)
m.update()
print("Objective set: minimize C_exp + C_build + C_equip")
print(f"  Piecewise tier vars (t1,t2,t3): {len(t1)} each")
print(f"  Piecewise slot vars (s1,s2,s3): {len(s1)} each")

Objective set: minimize C_exp + C_build + C_equip
  Piecewise tier vars (t1,t2,t3): 7739 each
  Piecewise slot vars (s1,s2,s3): 7739 each


In [65]:
# ── Constraint 1: Total slot coverage per desert ZIP ──────────────────────────
for z in desert_zips:
    facs      = fac_by_zip.get(z, [])
    locs      = locs_by_zip.get(z, [])
    total_def = int(desert_df.loc[z, 'total_slot_deficit'])
    m.addConstr(
        gp.quicksum(x_05[f] + x_512[f] for f in facs)
        + gp.quicksum(
            CAP_S * b['S'][l] + CAP_M * b['M'][l] + CAP_L * b['L'][l]
            for l in locs)
        >= total_def,
        name=f"total_cov_{z}"
    )

# ── Constraint 2: Under-5 slot coverage per desert ZIP ────────────────────────
for z in desert_zips:
    facs       = fac_by_zip.get(z, [])
    under5_def = int(desert_df.loc[z, 'under5_slot_deficit'])
    m.addConstr(
        gp.quicksum(x_05[f] for f in facs) + n_05[z]
        >= under5_def,
        name=f"under5_cov_{z}"
    )

# ── Constraint 3: New build slot accounting ────────────────────────────────────
for z in desert_zips:
    locs = locs_by_zip.get(z, [])
    m.addConstr(
        n_05[z] + n_512[z]
        <= gp.quicksum(
            CAP_S * b['S'][l] + CAP_M * b['M'][l] + CAP_L * b['L'][l]
            for l in locs),
        name=f"build_cap_{z}"
    )

# ── Constraint 4: New build under-5 cap ───────────────────────────────────────
for z in desert_zips:
    locs = locs_by_zip.get(z, [])
    m.addConstr(
        n_05[z]
        <= gp.quicksum(
            UNDER5_CAP_S * b['S'][l] + UNDER5_CAP_M * b['M'][l] + UNDER5_CAP_L * b['L'][l]
            for l in locs),
        name=f"under5_cap_{z}"
    )

# ── Constraint 5: At most one facility size per location ──────────────────────
for l in all_loc_ids:
    m.addConstr(
        b['S'][l] + b['M'][l] + b['L'][l] <= 1,
        name=f"one_size_{l}"
    )

# ── Constraint 6: Distance — forbidden pairs of potential locations ────────────
for (i, j) in forbidden_pairs:
    m.addConstr(
        gp.quicksum(b[s][i] for s in ['S','M','L'])
        + gp.quicksum(b[s][j] for s in ['S','M','L'])
        <= 1,
        name=f"dist_{i}_{j}"
    )

m.update()
print("Constraints added:")
print(f"  1. Total slot coverage      : {len(desert_zips)}")
print(f"  2. Under-5 coverage         : {len(desert_zips)}")
print(f"  3. New build slot cap       : {len(desert_zips)}")
print(f"  4. New build under-5 cap    : {len(desert_zips)}")
print(f"  5. One size per location    : {len(all_loc_ids)}")
print(f"  6. Distance forbidden pairs : {len(forbidden_pairs)}")
print(f"\nTotal constraints: {m.NumConstrs}")

Constraints added:
  1. Total slot coverage      : 179
  2. Under-5 coverage         : 179
  3. New build slot cap       : 179
  4. New build under-5 cap    : 179
  5. One size per location    : 27959
  6. Distance forbidden pairs : 86996

Total constraints: 185322


In [66]:
m.optimize()

if m.Status == GRB.OPTIMAL:
    total_cost  = m.ObjVal
    C_exp_val   = sum(
        (20_000 + 200  * cap_lookup[f]) / cap_lookup[f] * s1[f].X
      + (20_000 + 400  * cap_lookup[f]) / cap_lookup[f] * s2[f].X
      + (20_000 + 1000 * cap_lookup[f]) / cap_lookup[f] * s3[f].X
        for f in cap_lookup
    )
    C_build_val = sum(
        COST_S * b['S'][l].X + COST_M * b['M'][l].X + COST_L * b['L'][l].X
        for l in all_loc_ids
    )
    C_equip_val = EQUIP_COST * sum(
        sum(x_05[f].X for f in fac_by_zip.get(z, [])) + n_05[z].X
        for z in desert_zips
    )

    total_exp_05  = sum(x_05[f].X  for f in cap_lookup)
    total_exp_512 = sum(x_512[f].X for f in cap_lookup)
    new_S = sum(b['S'][l].X for l in all_loc_ids)
    new_M = sum(b['M'][l].X for l in all_loc_ids)
    new_L = sum(b['L'][l].X for l in all_loc_ids)

    print("=" * 55)
    print("           OPTIMAL SOLUTION FOUND")
    print("=" * 55)
    print(f"\n  Total Minimum Cost       : ${total_cost:>15,.2f}")
    print(f"  ├─ Expansion Cost        : ${C_exp_val:>15,.2f}")
    print(f"  ├─ Construction Cost     : ${C_build_val:>15,.2f}")
    print(f"  └─ Equipment Surcharge   : ${C_equip_val:>15,.2f}")
    print(f"\n  Expansion slots (0-5)   : {total_exp_05:>10,.0f}")
    print(f"  Expansion slots (5-12)  : {total_exp_512:>10,.0f}")
    print(f"  New facilities built    : {new_S+new_M+new_L:>10,.0f}")
    print(f"    ├─ Small  (100 slots) : {new_S:>10,.0f}")
    print(f"    ├─ Medium (200 slots) : {new_M:>10,.0f}")
    print(f"    └─ Large  (400 slots) : {new_L:>10,.0f}")

elif m.Status == GRB.INFEASIBLE:
    print("Model is INFEASIBLE")
    m.computeIIS()
    m.write("infeasible.ilp")
else:
    print(f"Solver status: {m.Status}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Academic license 2792667 - for non-commercial use only - registered to pc___@columbia.edu
Optimize a model with 185322 rows, 146147 columns and 1050775 nonzeros (Min)
Model fingerprint: 0xc34c9868
Model has 115012 linear objective coefficients
Variable types: 23217 continuous, 122930 integer (107094 binary)
Coefficient statistics:
  Matrix range     [1e-01, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e-01, 1e+04]

Presolve removed 111799 rows and 33576 columns (presolve time = 5s)...
Presolve removed 165792 rows and 85359 columns
Presolve time: 9.50s
Presolved: 19530 rows, 60788 columns, 219755 nonzeros
Variable types: 0 continuous, 60788 integer (59253 binary)
Found heuristic solution: objective 2.45138